# A second opinion on BIOMASS forest height
## Two BIOMASS forest-height products against GEDI and the airborne LiDAR

**ESA MAAP Hackathon 2026 — Challenge #5 "Canopy Clash"** · follow-up of notebook 01

**Where we are.** Notebook 01 tested the official BIOMASS L2A forest height (FH) at three LiDAR sites in Gabon:
- good in flat forest (RMSE 7–9 m against LVIS);
- too low on slopes **facing the radar** (foreshortening, up to −20 m);
- too high on savanna (16–23 m instead of 3–6 m);
- 6–8 m of random disagreement between two dates.

**A new candidate.** The BIOMASS-Coding2026 workshop folder contains another forest-height product for Gabon: **one geocoded height map per acquisition** (November 2025 – June 2026, about 100 m pixels) and a **mosaic** of all of them. It is said to be more reliable than the official product.

**We do not take that on trust.** We put it through exactly the same tests as the official product: same three sites, same pixels, same referee (NASA LVIS airborne LiDAR).

| Question | Test |
|---|---|
| 1. Pixel by pixel, which product is closer to the LiDAR? | accuracy by landscape stratum (Chapter 3) |
| 2. Are the errors random or systematic? | averaging test (Chapter 4) |
| 3. Does it also fail on slopes facing the radar? | error vs local incidence angle (Chapter 5) |
| 4. And on savanna? | short-vegetation test (Chapter 6) |
| 5. Do two dates agree? | repeatability (Chapter 7) |
| 6. Is it in the right place? | geolocation check (Chapter 8) |
| 7. **What does GEDI say, and can GEDI be trusted as referee?** | **GEDI vs LVIS, BIOMASS vs GEDI, inside and outside the LiDAR strips (Chapter 9)** |

## 1 — The two products

| | Official L2A FH | Workshop FH ("FH_inv") |
|---|---|---|
| Where | ESA MAAP STAC catalogue, processor 4.4.4 | Examples_and_Data/Workshops/BIOMASS-Coding2026/Gabon/fh_gabon_geocoded |
| One file per | stack of acquisitions of one track (INT phase) | acquisition date (+ one mosaic of all dates) |
| Pixel | ≈ 0.0009° (≈ 100 m); 200 m resolution | ≈ 0.0009° (≈ 100 m) |
| Metadata | orbit, track, footprint, quality layer, forest mask | **none**: most files have no CRS, no orbit, no quality layer |

Because the workshop files carry no metadata, we reconstruct what we need from the data itself:
- **Coordinates:** the pixel grid is longitude/latitude, so we read the files as EPSG:4326.
- **Orbit direction:** from the acquisition time in the folder name. 05 UTC passes are ascending and 17 UTC passes descending. We checked this against the official products over the same sites, which record their direction.
- **Footprint, heading, incidence and local incidence angle (LIA):** from the edges of the valid data, with exactly the same code used for the official product.

Every product is then put on the **same grid as notebook 01**, so every pixel is compared with the same LiDAR pixel.

In [ ]:
# Setup
import importlib.util, subprocess, sys, json, os, warnings
from pathlib import Path
if importlib.util.find_spec("pptx") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-pptx"], check=True)

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle, Polygon
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = next((c for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (c / "src" / "__init__.py").exists() and (c / "config" / "sites.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the topic folder (it needs src/ and config/).")
sys.path.insert(0, str(ROOT)); importlib.invalidate_caches()
from src.load.workshop import inventory, mosaic_on_grid
from src.process.site import load_site, site_passes, mosaic_site
from src.process.aggregate import at_scale, gedi_blocks, best_gedi
from src.evaluate import accuracy as acc
from src.evaluate.multiscale import scale_curve, best_shift, block_comparison

CFG = json.loads((ROOT / "config" / "sites.json").read_text()); SITES = CFG["sites"]
WS_DIR = Path(CFG["workshop_fh_dir"]).expanduser()   # workshop FH folder on MAAP
FIG = ROOT / "results" / "workshop_fh"; FIG.mkdir(parents=True, exist_ok=True)
C_L2A, C_WS, C_MOS, C_LVIS = "#2a78d6", "#c2457a", "#7a2048", "#eb6834"
C_ASC, C_DESC = "#6250d6", "#eda100"
PRODUCTS = {"L2A": C_L2A, "Workshop": C_WS, "Mosaic": C_MOS}
HEIGHT_CMAP = LinearSegmentedColormap.from_list("height", ["#f4f1e6", "#9fcf8f", "#2e8b3e", "#0b3d1f"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "axes.titlesize": 11, "axes.titleweight": "bold"})
FACTS, FIGS = {}, {}

def savefig(fig, name):
    p = FIG / f"{name}.png"; fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); FIGS[name] = p; return p

print("Project:", ROOT)
print("Workshop products:", WS_DIR)

In [ ]:
# Every product on the same grid as notebook 01 (the LiDAR grid of each site)
kw = dict(swath_incidence=tuple(CFG["swath_incidence_deg"]), min_coverage=CFG["lvis_min_coverage_fraction"],
          min_pass_coverage=CFG.get("min_pass_lidar_coverage", 0.2))
D = {k: load_site(k, s, ROOT, **kw) for k, s in SITES.items()}                                     # official L2A
INV = inventory(WS_DIR)                                                                                # workshop files
W = {k: site_passes(S, INV, kw["swath_incidence"], kw["min_pass_coverage"]) for k, S in D.items()}  # workshop, per date
M = {k: mosaic_site(S, mosaic_on_grid(S, WS_DIR)) for k, S in D.items()}                              # workshop mosaic
KEYS = list(D)
PROD = {"L2A": D, "Workshop": W, "Mosaic": M}
print(f"{len(INV)} workshop acquisitions, {INV.date.min():%d %b %Y} – {INV.date.max():%d %b %Y}")

In [ ]:
# Which workshop acquisitions cover our LiDAR sites, and which official pass do they correspond to?
pairs = []
for k in KEYS:
    for p in W[k]["all_passes"]:
        same = [q for q in D[k]["all_passes"] if q["orbit"] == p["orbit"]]
        q = min(same, key=lambda q: abs((q["date"] - p["date"]).days)) if same else None
        pairs.append({"site": D[k]["name"], "workshop pass": p["label"], "frames": len(p["frames"]),
                      "LiDAR area covered [%]": round(100 * p["lidar_coverage"]),
                      "used": "yes" if p in W[k]["scenes"] else "no (too small)",
                      "closest official pass, same direction": q["label"] if q else "-",
                      "days apart": (q["date"] - p["date"]).days if q else None})
PAIRS = pd.DataFrame(pairs); display(PAIRS)

The workshop acquisitions fall **a few days before an official L2A pass in the same direction**, so the two products look at the same forest at almost the same time. Differences between them come from the processing, not from the forest.

In [ ]:
# Map: the mosaic over Gabon and the footprint of every acquisition
with rasterio.open(WS_DIR / "tif-mosaic" / "mosaic_FH_100m.tif") as src:
    f = 10
    mos = src.read(1, out_shape=(src.height // f, src.width // f)).astype(float); b = src.bounds
mos[(mos < 0) | (mos == -9999)] = np.nan
ext = [b.left, b.right, b.bottom, b.top]
fig, ax = plt.subplots(1, 2, figsize=(13, 7.5))
im = ax[0].imshow(mos, extent=ext, cmap=HEIGHT_CMAP, norm=Normalize(0, 60), interpolation="nearest")
for _, r in INV.iterrows():
    ax[1].add_patch(Polygon(np.asarray(r.footprint), closed=True, fill=False, lw=1, alpha=.8,
                            ec=C_ASC if r.orbit == "ascending" else C_DESC))
for a in ax:
    for k in KEYS:
        w_, s_, e_, n_ = SITES[k]["bbox"]
        a.add_patch(Rectangle((w_, s_), e_ - w_, n_ - s_, fill=False, ec="#d03b3b", lw=1.8))
        a.annotate(SITES[k]["name"], (e_, n_), xytext=(4, 2), textcoords="offset points", color="#d03b3b", fontsize=9, fontweight="bold")
    a.set(xlim=ext[:2], ylim=ext[2:], aspect="equal", xlabel="Longitude [°]", ylabel="Latitude [°]")
ax[0].set_title("Workshop FH mosaic (red: LiDAR sites)")
ax[1].set_title("Acquisition footprints (purple ascending, yellow descending)")
fig.colorbar(im, ax=ax[0], shrink=.6, label="Height [m]")
fig.tight_layout(); savefig(fig, "01_product"); plt.show()

## 2 — First look: the same forest in three maps

Same colour scale (0–60 m) and same window as notebook 01. Left to right: the LiDAR reference, the official product, the workshop product for one date, and the workshop mosaic.

In [ ]:
fig, ax = plt.subplots(len(KEYS), 4, figsize=(17, 4.4 * len(KEYS)), squeeze=False)
norm = Normalize(0, 60)
for i, k in enumerate(KEYS):
    S = D[k]; win, ext = S["win"], S["win_extent"]
    panels = [(S["lvis"], "LVIS 2016 (reference)"),
              (S["best"]["h"] if S["best"] else None, f"Official L2A ({S['best']['label']})" if S["best"] else "L2A: none"),
              (W[k]["best"]["h"] if W[k]["best"] else None, f"Workshop ({W[k]['best']['label']})" if W[k]["best"] else "Workshop: none"),
              (M[k]["best"]["h"], "Workshop mosaic")]
    for j, (arr, title) in enumerate(panels):
        a = ax[i, j]; a.set_facecolor("#ecebe6")
        if arr is not None:
            im = a.imshow(arr[win], extent=ext, cmap=HEIGHT_CMAP, norm=norm, interpolation="nearest")
        a.set_title(f"{S['name']}\n{title}", fontsize=9.5); a.set_xticks([]); a.set_yticks([])
fig.colorbar(im, ax=ax, shrink=.5, label="Height [m]")
savefig(fig, "02_first_look"); plt.show()

The official map is **smooth**: its values stay in a narrow band. The workshop map is **grainy**: neighbouring pixels jump between very low and very high values. The histograms below show the difference in numbers.

In [ ]:
bins = np.arange(0, 82, 2)
fig, ax = plt.subplots(1, len(KEYS), figsize=(5.4 * len(KEYS), 4), sharey=True, squeeze=False)
rows = []
for a, k in zip(ax[0], KEYS):
    S = D[k]; m = (S["cover"] == acc.FOREST) & np.isfinite(S["lvis"])
    a.hist(S["lvis"][m], bins, density=True, histtype="step", lw=2.4, color=C_LVIS, label="LVIS")
    rows.append({"site": S["name"], "product": "LVIS", **dict(zip(["p5", "median", "p95"], np.nanpercentile(S["lvis"][m], [5, 50, 95])))})
    for name, P in PROD.items():
        if not P[k]["scenes"]:
            continue
        v = np.concatenate([p["h"][m] for p in P[k]["scenes"]]); v = v[np.isfinite(v)]
        a.hist(v, bins, density=True, histtype="step", lw=1.7, color=PRODUCTS[name], label=name)
        rows.append({"site": S["name"], "product": name, **dict(zip(["p5", "median", "p95"], np.percentile(v, [5, 50, 95])))})
    a.set(title=f"{S['name']}: forest under the LiDAR", xlabel="Height [m]")
ax[0, 0].set_ylabel("Density"); ax[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout(); savefig(fig, "03_histograms"); plt.show()
SPREAD = pd.DataFrame(rows); SPREAD["p95 − p5"] = SPREAD.p95 - SPREAD.p5
display(SPREAD.round(1))

**Reading the histograms.**
- **Official L2A** is **too narrow**. It squeezes every forest into a few metres around 28–35 m, so tall forest comes out too low (the "compression" seen in notebook 01).
- **Workshop** is **too wide**. Its median is close to the LiDAR, but 5% of forest pixels are below 5 m and the tail goes beyond 60 m, which is impossible for these forests.

Two different kinds of error: one product is *biased but calm*, the other *centred but noisy*. The next two chapters measure which one matters more.

## 3 — Pixel by pixel: which product is closer to the LiDAR?

Same strata as notebook 01. Errors are **product − LVIS** (negative = too low); median over the passes of each site.

In [ ]:
ACC = {name: acc.accuracy_table(list(P.values())) for name, P in PROD.items()}
SUM = pd.concat([acc.site_summary(a, "LVIS").assign(product=name) for name, a in ACC.items()], ignore_index=True)
SUM.to_csv(FIG / "accuracy_by_stratum_100m.csv", index=False)
display(SUM.pivot_table(index=["site", "stratum"], columns="product", values=["rmse", "bias", "r"], sort=False).round(2))

order = ["forest, flat (<5°)", "forest, moderate (5–15°)", "forest, steep (≥15°)", "savanna"]
fig, ax = plt.subplots(1, len(KEYS), figsize=(5.4 * len(KEYS), 4.2), sharey=True, squeeze=False)
x = np.arange(len(order)); wb = .27
for a, k in zip(ax[0], KEYS):
    for i, (name, c) in enumerate(PRODUCTS.items()):
        t = SUM[(SUM["product"] == name) & (SUM.site == D[k]["name"])].set_index("stratum").reindex(order)
        a.bar(x + (i - 1) * wb, t.rmse, wb, color=c, label=name)
    a.set_xticks(x, ["flat", "moderate", "steep", "savanna"])
    a.set_title(f"{D[k]['name']}: RMSE vs LVIS, 100 m pixels")
ax[0, 0].set_ylabel("RMSE [m]"); ax[0, 0].legend(frameon=False)
fig.tight_layout(); savefig(fig, "04_rmse_by_stratum"); plt.show()

At the native 100 m pixel the answer is clear: **the official product wins everywhere**. The workshop RMSE is roughly **twice** as large, and the mosaic does not help, because it is built from the same noisy maps.

But RMSE mixes two things: a **systematic** part (the bias, always the same direction) and a **random** part (the noise, different for every pixel). They behave very differently when we look at the forest at a coarser scale.

## 4 — Noise or bias? The averaging test

We average both the product **and** the LiDAR over windows of 3×3, 5×5, 9×9 and 15×15 pixels (about 300 m to 1.4 km) and recompute the forest error.
- **Random noise** averages out: the RMSE falls quickly with the window.
- **Systematic error** does not average out: it stays as it is.

Since RMSE² = bias² + SD², the bottom row shows the systematic part alone.

In [ ]:
SCALE = pd.concat([scale_curve(P[k], name) for k in KEYS for name, P in PROD.items()], ignore_index=True)
SCALE.to_csv(FIG / "error_vs_scale.csv", index=False)
display(SCALE.pivot_table(index=["site", "size_m"], columns="product", values=["rmse", "bias"], sort=False).round(2))
fig, ax = plt.subplots(2, len(KEYS), figsize=(5.4 * len(KEYS), 7.6), sharex=True, squeeze=False)
for j, k in enumerate(KEYS):
    t = SCALE[SCALE.site == D[k]["name"]]
    for name, c in PRODUCTS.items():
        u = t[t["product"] == name]
        ax[0, j].plot(u.size_m, u.rmse, "o-", color=c, lw=2, label=name)
        ax[1, j].plot(u.size_m, u.bias.abs(), "o-", color=c, lw=2, label=name)
    ax[0, j].set_title(f"{D[k]['name']}: total error (RMSE)"); ax[1, j].set_title(f"{D[k]['name']}: systematic part (|bias|)")
    ax[1, j].set_xlabel("Averaging window [m]")
    for a in ax[:, j]:
        a.set_ylim(0, None); a.grid(axis="y", color="#ecebe6")
ax[0, 0].set_ylabel("RMSE vs LVIS [m]"); ax[1, 0].set_ylabel("|bias| vs LVIS [m]"); ax[0, 0].legend(frameon=False)
fig.tight_layout(); savefig(fig, "05_error_vs_scale"); plt.show()

In [ ]:
# Compare again by stratum, at about 500 m (5 x 5 pixels), the scale where the two curves meet
K = 5
SUM5 = pd.concat([acc.site_summary(acc.accuracy_table([at_scale(P[k], K) for k in KEYS]), "LVIS").assign(product=name)
                  for name, P in PROD.items()], ignore_index=True)
SUM5.to_csv(FIG / "accuracy_by_stratum_500m.csv", index=False)
display(SUM5.pivot_table(index=["site", "stratum"], columns="product", values=["rmse", "bias", "r"], sort=False).round(2))

**Reading the averaging test.**
- **Workshop:** almost all of its error is **noise**. Its curve drops steeply. In the hilly sites (Mabounie, Lopé) it reaches the official product at about 300–500 m and is **better** beyond that, with a smaller bias. In flat Rabi it gets close but never quite overtakes.
- **Official L2A:** the curve is flat, because most of its error is **systematic** (the compression and the slope bias). Averaging cannot remove that.

So "which product is more reliable?" depends on the scale you need:
- for a **map at 100 m**, the official product;
- for **forest height over areas of a few hundred metres or more** (for example to calibrate a biomass map), the workshop product averaged over 5×5 pixels or more.

## 5 — Slopes: does the workshop product also suffer from foreshortening?

Notebook 01 found that the official product is **10–20 m too low** where the slope faces the radar (small local incidence angle). Here the same test is run for both products (100 m pixels). Solid lines: official L2A. Dashed lines: workshop.

In [ ]:
LIA = pd.concat([pd.DataFrame()] + [acc.error_vs_lia(P[k]).assign(product=name)
                                    for k in KEYS for name, P in [("L2A", D), ("Workshop", W)] if P[k]["scenes"] and P[k]["lvis"] is not None],
                ignore_index=True)
LIA.to_csv(FIG / "error_vs_lia.csv", index=False)
fig, ax = plt.subplots(1, len(KEYS), figsize=(5.4 * len(KEYS), 4.3), sharey=True, squeeze=False)
for a, k in zip(ax[0], KEYS):
    t = LIA[LIA.site == D[k]["name"]]
    for (prod, ls, mk) in [("L2A", "-", "o"), ("Workshop", "--", "s")]:
        for orb, c in [("ascending", C_ASC), ("descending", C_DESC)]:
            u = t[(t["product"] == prod) & (t.orbit == orb)]
            if len(u):
                a.plot(u.lia_mid, u.bias, ls=ls, marker=mk, color=c, lw=2, ms=5, label=f"{prod} {orb}")
    a.axhline(0, color="#c3c2b7", lw=1)
    a.set(title=f"{D[k]['name']}: forest error vs LIA", xlabel="Local incidence angle [°]")
ax[0, 0].set_ylabel("Product − LVIS bias [m]"); ax[0, 0].legend(frameon=False, fontsize=7.5)
fig.tight_layout(); savefig(fig, "06_error_vs_lia"); plt.show()

rows = []
for (prod, site, orb), t in LIA.groupby(["product", "site", "orbit"]):
    low = t[t.lia_mid < 20]; mid = t[(t.lia_mid >= 25) & (t.lia_mid < 35)]; high = t[t.lia_mid >= 40]
    if len(low) and len(mid):
        rows.append(dict(product=prod, site=site, orbit=orb,
                         facing_radar=np.average(low.bias, weights=low.n), radar_flat=np.average(mid.bias, weights=mid.n),
                         facing_away=np.average(high.bias, weights=high.n) if len(high) else np.nan))
TERRAIN = pd.DataFrame(rows)
if len(TERRAIN):
    TERRAIN["facing radar − flat [m]"] = TERRAIN.facing_radar - TERRAIN.radar_flat
    TERRAIN["facing away − flat [m]"] = TERRAIN.facing_away - TERRAIN.radar_flat
    TERRAIN["total swing [m]"] = TERRAIN.facing_away - TERRAIN.facing_radar
display(TERRAIN.round(1))


**Reading it.** The table summarises each curve with three numbers, all relative to terrain that looks flat to the radar (LIA 25–35°):
- **facing radar − flat**: extra error on slopes facing the radar (LIA < 20°). Negative = too low (foreshortening).
- **facing away − flat**: extra error on slopes facing away (LIA ≥ 40°).
- **total swing**: how much the error changes from one side of a hill to the other. Zero would mean the product ignores the viewing geometry.

**What we see.** On slopes **facing the radar** the workshop product drops much less than the official one (a few metres instead of 6–11 m). But it is **not free of terrain effects**: its error keeps rising with the LIA and becomes an **overestimation** on slopes facing away, up to about +15 m at Mabounie. The official product is the opposite: strongly too low on the radar side, roughly flat on the far side. Both products still "see the hills"; they just lean to different sides.

## 6 — Savanna: does it see short vegetation?

Same test as notebook 01: vegetation below 10 m. *Recall* = share of truly short pixels (LiDAR) that the product also calls short; *precision* = share of the product's "short" pixels that are truly short.

In [ ]:
SV = pd.concat([pd.DataFrame()] + [acc.short_vegetation(P[k], CFG["low_height_m"]).assign(product=name) for k in KEYS for name, P in PROD.items()],
               ignore_index=True)
SV.to_csv(FIG / "short_vegetation.csv", index=False)
display(SV.pivot_table(index=["site", "cover"], columns="product",
                       values=["pixels", "lvis_median", "biomass_median", "recall", "precision"], sort=False).round(1))

In this table "biomass_median" is the median height of **the product in that column**. Look at the savanna rows: the LiDAR says 3–6 m. Watch the pixel counts too. A product that leaves savanna pixels empty (masked) cannot be wrong there, but it also says nothing about them.

## 7 — Do two dates agree?

Each product is compared **with itself** on forest pixels, for every pair of dates, at 100 m and at 500 m. The official product has several passes on the same track. For the workshop product we do not know the track, so all pairs with the same direction are grouped as "same direction".

In [ ]:
def rep(P, k, scale=1):
    S = P[k] if scale == 1 else at_scale(P[k], scale)
    if len(S["scenes"]) < 2:
        return pd.DataFrame()
    return acc.repeatability(S, mask=(D[k]["cover"] == acc.FOREST))

REP = pd.concat([pd.DataFrame()] + [rep(P, k, s).assign(product=name, scale=f"{s * 100} m")
                                    for k in KEYS for name, P in [("L2A", D), ("Workshop", W)] for s in (1, 5)], ignore_index=True)
REP["geometry"] = REP.geometry.replace({"same track": "same direction"})
REP = REP[REP.n >= 1000]
REP.to_csv(FIG / "repeatability.csv", index=False)
display(REP[REP.scale == "100 m"][["product", "site", "a", "b", "days", "geometry", "n", "bias", "rmse", "r"]].round(2))
REP_SUM = REP.groupby(["product", "scale", "geometry"]).agg(pairs=("rmse", "size"), median_rmse=("rmse", "median"),
                                                          median_r=("r", "median")).reset_index()
display(REP_SUM.round(2))

## 8 — Is it in the right place? (geolocation check)

We shift each map by up to ±5 pixels (≈ ±500 m) and look for the shift that best matches the LiDAR. A well-geolocated product gives its best match at zero shift; a shift of one pixel is within the noise.

In [ ]:
rows = []
for k in KEYS:
    S = D[k]; m = S["cover"] == acc.FOREST
    for name, P in PROD.items():
        if P[k]["best"] is None or S["lvis"] is None:
            continue
        r = best_shift(P[k]["best"]["h"], S["lvis"], m)
        rows.append({"site": S["name"], "product": name, "pass": P[k]["best"]["label"], **r,
                     "shift [m]": int(round(np.hypot(r["shift_rows"], r["shift_cols"]) * S["grid"].res * 111320, -1))})
GEO = pd.DataFrame(rows); GEO.to_csv(FIG / "geolocation.csv", index=False)
display(GEO.round(3))

**Reading it.** A shift of one pixel (≈ 90–130 m) is within the noise of this test: both products are essentially in the right place. The only larger shift is the official Mabounie pass (≈ 290 m, correlation 0.40 → 0.50). It is a single pass, so it is a lead to check on other dates, not a conclusion.

## 9 — And GEDI? The comparison the challenge asks for

Up to now the referee was the airborne LiDAR. But LVIS exists only on a few strips flown in 2016. Almost everywhere else, the only independent canopy height is **GEDI**, the LiDAR on the International Space Station (2019–2025). So the real questions of the challenge are:
- **(a)** Is GEDI a good referee? We check it against LVIS.
- **(b)** What does BIOMASS look like against GEDI, and does GEDI give the same verdict as LVIS?
- **(c)** Do the results hold **outside** the LiDAR strips, where only GEDI exists?

**How we compare.** A GEDI shot is a 25 m footprint, and the shots lie along tracks, so a 100 m pixel usually contains one shot at most. Single-pixel comparisons are therefore very noisy. We compare **blocks of k×k pixels** (from 100 m to ≈ 1 km):
- product = mean of the block;
- LVIS = mean of the block;
- GEDI = median of the shots in the block (at least 3 shots for blocks larger than one pixel).

Forest blocks only (≥ 80% forest). The GEDI shots were already filtered at download: quality_flag = 1, degrade_flag = 0, sensitivity ≥ 0.95.

In [ ]:
C_GEDI = "#1baf7a"
BC = block_comparison(D, {"L2A": D, "Workshop": W, "Mosaic": M})
BC.to_csv(FIG / "gedi_block_comparison.csv", index=False)
print(f"{len(BC)} rows; block sizes: {sorted(BC.size_m.unique())} m")

### 9a — Is GEDI a good referee?

In [ ]:
display(BC[BC.reference.eq("LVIS") & BC.estimate.str.startswith("GEDI")]
        .pivot_table(index=["site", "estimate"], columns="size_m", values=["n", "bias", "rmse"], sort=False).round(2))
fig, ax = plt.subplots(1, len(KEYS), figsize=(5.4 * len(KEYS), 3.8), sharey=True, squeeze=False)
bins = np.arange(-40, 41, 2)
for a, k in zip(ax[0], KEYS):
    S = D[k]; f = (S["cover"] == acc.FOREST) & np.isfinite(S["lvis"])
    for shots, lab, c in ((S["gedi_shots"], "all shots", C_GEDI), (best_gedi(S["gedi_shots"]), "power beams, night", "#0b6b47")):
        G, _ = gedi_blocks(S, 1, shots)
        d = (G - S["lvis"])[f & np.isfinite(G)]
        a.hist(d, bins, density=True, histtype="step", lw=2, color=c, label=f"{lab} (median {np.median(d):+.1f} m)")
    a.axvline(0, color="#c3c2b7", lw=1)
    a.set(title=f"{S['name']}: GEDI − LVIS, forest pixels", xlabel="Height difference [m]")
    a.legend(frameon=False, fontsize=8)
ax[0, 0].set_ylabel("Density")
fig.tight_layout(); savefig(fig, "08_gedi_vs_lvis"); plt.show()

**Reading it.**
- **GEDI is lower than LVIS** by about 2–5 m in these forests. Part of it is GEDI itself: in tall, dense tropical canopy the weak echo from the top is easy to miss. Part is time: LVIS is from 2016, GEDI from 2019–2025. The **power beams at night**, which penetrate best, reduce the gap a lot.
- **Per pixel the scatter is large** (8–10 m RMSE) and it stays at 5–8 m even over ≈ 1 km blocks. GEDI's geolocation (≈ 10 m), its small footprint and real forest change since 2016 all contribute.

So GEDI is a **good referee for averages over many shots, a noisy one for single pixels, and it sits a few metres low**. Everything BIOMASS is compared with in the next step inherits this.

### 9b — BIOMASS against GEDI, next to BIOMASS against LVIS

Same blocks, inside the LiDAR area, so both referees judge the same forest. **Solid lines**: against LVIS. **Dashed lines**: against GEDI. **Dotted green**: GEDI against LVIS (the referee's own error).

In [ ]:
fig, ax = plt.subplots(2, len(KEYS), figsize=(5.4 * len(KEYS), 7.8), sharex=True, squeeze=False)
for j, k in enumerate(KEYS):
    t = BC[(BC.site == D[k]["name"]) & (BC.area == "LiDAR area")]
    for name in ("L2A", "Workshop"):
        for ref, ls in (("LVIS", "-"), ("GEDI", "--")):
            u = t[(t.estimate == name) & (t.reference == ref)].sort_values("size_m")
            ax[0, j].plot(u.size_m, u.rmse, ls=ls, marker="o", ms=4, color=PRODUCTS[name], lw=2, label=f"{name} vs {ref}")
            ax[1, j].plot(u.size_m, u.bias, ls=ls, marker="o", ms=4, color=PRODUCTS[name], lw=2, label=f"{name} vs {ref}")
    u = t[t.estimate == "GEDI (all shots)"].sort_values("size_m")
    ax[0, j].plot(u.size_m, u.rmse, ls=":", marker="s", ms=4, color=C_GEDI, lw=2, label="GEDI vs LVIS")
    ax[1, j].plot(u.size_m, u.bias, ls=":", marker="s", ms=4, color=C_GEDI, lw=2, label="GEDI vs LVIS")
    ax[0, j].set_title(f"{D[k]['name']}: RMSE"); ax[1, j].set_title(f"{D[k]['name']}: bias (estimate − reference)")
    ax[0, j].set_ylim(0, None); ax[1, j].axhline(0, color="#c3c2b7", lw=1); ax[1, j].set_xlabel("Block size [m]")
ax[0, 0].set_ylabel("RMSE [m]"); ax[1, 0].set_ylabel("Bias [m]"); ax[0, 0].legend(frameon=False, fontsize=7.5)
fig.tight_layout(); savefig(fig, "09_gedi_vs_lvis_referee"); plt.show()

t = BC[(BC.area == "LiDAR area") & BC.estimate.isin(["L2A", "Workshop", "Mosaic"]) & BC.k.isin([1, BC.k.max()])]
display(t.pivot_table(index=["site", "estimate"], columns=["size_m", "reference"], values=["bias", "rmse"], sort=False).round(2))

**Reading it.**
- **Bias: GEDI makes BIOMASS look better than it is.** Against GEDI the official product is almost unbiased (within a few metres of zero), against LVIS it is 4–5 m too low. The difference is GEDI's own low bias (dotted line). An "unbiased against GEDI" result therefore does **not** mean an unbiased product: both GEDI and BIOMASS miss the top of the tallest canopy.
- **RMSE against GEDI is always higher than against LVIS**, because GEDI adds its own noise. Even a perfect BIOMASS map could not get below the dotted line.
- **The verdict between the two BIOMASS products is the same with either referee**: the official one is better at 100 m, the workshop one catches up when averaged, and at ≈ 1 km it ties or wins in the hilly sites. **GEDI is enough to reach the right ranking**, as long as we aggregate and remember its bias.

### 9c — Beyond the LiDAR strips

GEDI covers the whole site box, not only the LVIS strips. If the numbers outside the strips are similar, our conclusions are not a peculiarity of the few kilometres flown by the aircraft.

In [ ]:
t = BC[(BC.reference == "GEDI") & BC.estimate.isin(["L2A", "Workshop"]) & BC.k.isin([5, BC.k.max()])]
display(t.pivot_table(index=["site", "estimate", "size_m"], columns="area", values=["n", "bias", "rmse"], sort=False).round(2))

Outside the strips there are **many more blocks** and the errors are of the same size as inside, so the picture holds for the whole site. Where the bias changes between inside and outside (for example the workshop product at Lopé, a few metres higher outside the strips), the landscape outside is different: more forest–savanna edges and steeper hills.

## 10 — Verdict

The numbers below are generated from the tables above.

In [ ]:
L = []
s1 = SCALE[SCALE.window == 1].pivot(index="site", columns="product", values="rmse")
L.append("Pixel by pixel (100 m, forest): RMSE against LiDAR " +
         "; ".join(f"{s} L2A {r['L2A']:.1f} m vs workshop {r['Workshop']:.1f} m" for s, r in s1.iterrows()) +
         f". At full resolution the workshop product is {np.median(s1.Workshop / s1.L2A):.1f}× noisier than the official one.")
w1 = SCALE[SCALE.window == 1].assign(noise=lambda t: 100 * t.sd ** 2 / t.rmse ** 2)
sh = w1.pivot(index="site", columns="product", values="noise"); bl = w1[w1["product"] == "L2A"].bias.abs()
L.append(f"Share of the squared error that is random noise at 100 m: workshop {sh.Workshop.min():.0f}–{sh.Workshop.max():.0f}%, "
         f"official {sh.L2A.min():.0f}–{sh.L2A.max():.0f}%. The official product keeps a {bl.min():.1f}–{bl.max():.1f} m bias "
         "that does not average out (bottom row of the averaging figure).")
big = SCALE.window.max(); sb = SCALE[SCALE.window == big].pivot(index="site", columns="product", values="rmse")
cross = []
for s in s1.index:
    t = SCALE[SCALE.site == s].pivot(index="size_m", columns="product", values="rmse")
    ok = t[t.Workshop <= t.L2A]
    cross.append(f"{s} from ≈{int(ok.index.min())} m" if len(ok) else f"{s} never (up to {int(t.index.max())} m)")
L.append("Averaged over larger windows the noise disappears and the workshop product catches up: it beats the official one at " +
         ", ".join(cross) + ". At " + f"≈{int(SCALE[SCALE.window == big].size_m.median())} m: " +
         "; ".join(f"{s} L2A {r['L2A']:.1f} m vs workshop {r['Workshop']:.1f} m" for s, r in sb.iterrows()) + ".")
if len(TERRAIN):
    for (site, orb), t in TERRAIN.groupby(["site", "orbit"]):
        t = t.set_index("product")
        if {"L2A", "Workshop"} <= set(t.index):
            L.append(f"Slopes ({site}, {orb}), extra error relative to radar-flat terrain: facing the radar "
                     f"{t.loc['L2A', 'facing radar − flat [m]']:+.1f} m (official) vs {t.loc['Workshop', 'facing radar − flat [m]']:+.1f} m (workshop); "
                     f"facing away {t.loc['L2A', 'facing away − flat [m]']:+.1f} m vs {t.loc['Workshop', 'facing away − flat [m]']:+.1f} m.")
sv = SV[SV.cover == "savanna"]
for site, t in sv.groupby("site"):
    t = t.set_index("product")
    if "Workshop" in t.index and "L2A" in t.index and t.loc["L2A", "pixels"] >= 300:
        L.append(f"Savanna ({site}): LiDAR {t.loc['L2A', 'lvis_median']:.1f} m; official {t.loc['L2A', 'biomass_median']:.1f} m "
                 f"(recall {t.loc['L2A', 'recall']:.0f}%), workshop {t.loc['Workshop', 'biomass_median']:.1f} m "
                 f"(recall {t.loc['Workshop', 'recall']:.0f}%, {t.loc['Workshop', 'pixels']:.0f} of {t.loc['L2A', 'pixels']:.0f} pixels valid).")
if len(REP_SUM):
    r = REP_SUM[REP_SUM.geometry == "same direction"].set_index(["product", "scale"]).median_rmse
    L.append("Two dates, same direction (median RMSE between dates): " +
             ", ".join(f"{p} {r[(p, s)]:.1f} m at {s}" for p, s in r.index) + ".")
if len(GEO):
    L.append("Geolocation: best-matching shift " + ", ".join(f"{g.site} {g['product']} {g['shift [m]']} m" for _, g in GEO.iterrows()) +
             " (one pixel ≈ 100 m).")
g1 = BC[(BC.estimate == "GEDI (all shots)") & (BC.k == 1)]
gb = BC[(BC.estimate == "GEDI (power beams, night)") & (BC.k == 1)]
L.append(f"GEDI as referee (forest pixels): GEDI − LVIS bias {g1.bias.min():+.1f} to {g1.bias.max():+.1f} m (power beams at night "
         f"{gb.bias.min():+.1f} to {gb.bias.max():+.1f} m), RMSE {g1.rmse.min():.1f}–{g1.rmse.max():.1f} m per pixel.")
a = BC[(BC.estimate == "L2A") & (BC.area == "LiDAR area") & (BC.k == 1)].pivot(index="site", columns="reference", values="bias")
L.append("Official L2A bias against GEDI " + ", ".join(f"{s} {r['GEDI']:+.1f} m" for s, r in a.iterrows()) +
         " but against LVIS " + ", ".join(f"{s} {r['LVIS']:+.1f} m" for s, r in a.iterrows()) +
         ": GEDI's own low bias hides the underestimation of BIOMASS.")
kb = BC.k.max(); b = BC[(BC.reference == "GEDI") & (BC.area == "LiDAR area") & (BC.k == kb)].pivot(index="site", columns="estimate", values="rmse")
L.append(f"Against GEDI at ≈{int(BC[BC.k == kb].size_m.median())} m: " + "; ".join(f"{s} L2A {r['L2A']:.1f} m vs workshop {r['Workshop']:.1f} m" for s, r in b.iterrows()) +
         ". Broadly the same ranking as with LVIS: GEDI is enough to compare BIOMASS products if the data are aggregated.")
VERDICT = L
display(Markdown("\n".join(f"{i}. {l}" for i, l in enumerate(L, 1))))

**What it means.**
- **"More reliable" is true only at the right scale.** Pixel by pixel the workshop product is noisier, so for a 100 m map the official L2A is still the better choice.
- **Its errors are random rather than systematic.** That is good news: random noise can be removed by averaging (over space or over dates), a systematic bias cannot. Averaged over ≈ 500 m – 1 km, the workshop product matches or beats the official one in the hilly sites; in flat Rabi the two end up equivalent.
- **Terrain still matters, in a different way.** On slopes facing the radar the workshop product drops much less than the official one, but it overestimates slopes facing away. Neither product is free of the viewing geometry, so the ascending + descending combination proposed after notebook 01 is useful for both.
- **The two products fail in opposite ways.** The official one is calm but compressed and biased on slopes facing the radar; the workshop one is centred but noisy. This suggests the official processing smooths (or constrains) the inversion more strongly, trading noise for bias.
- **Caveat.** We do not know exactly how the workshop product was made (algorithm, number of acquisitions, filtering, masks). Before using it, that documentation is needed. Our tests say *how* it behaves, not *why*.

## 11 — Automatic report

The cell below builds a short slide deck from the figures and numbers of this notebook: results/Workshop_FH_vs_L2A.pptx.

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from PIL import Image

INK, MUTED, ACCENT = RGBColor(0x0B, 0x0B, 0x0B), RGBColor(0x52, 0x51, 0x4E), RGBColor(0xC2, 0x45, 0x7A)
prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5); BLANK = prs.slide_layouts[6]

def textbox(slide, x, y, w, h, paras, size=16, color=INK, bold_first=False):
    tf = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)).text_frame; tf.word_wrap = True
    for i, t in enumerate(paras):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = t; p.font.size = Pt(size); p.font.color.rgb = color; p.font.bold = bold_first and i == 0; p.space_after = Pt(8)

def header(slide, title, kicker=None):
    bar = slide.shapes.add_shape(1, 0, 0, prs.slide_width, Inches(0.12)); bar.fill.solid(); bar.fill.fore_color.rgb = ACCENT; bar.line.fill.background()
    textbox(slide, 0.5, 0.3, 12.3, 0.8, [title], size=28, bold_first=True)
    if kicker: textbox(slide, 0.5, 0.95, 12.3, 0.5, [kicker], size=14, color=MUTED)

def picture(slide, path, x, y, mw, mh):
    w, h = Image.open(path).size; s = min(mw / w, mh / h)
    slide.shapes.add_picture(str(path), Inches(x + (mw - w * s) / 2), Inches(y + (mh - h * s) / 2), Inches(w * s), Inches(h * s))

def fig_slide(title, kicker, key, bullets, full=True):
    if key not in FIGS: return
    s = prs.slides.add_slide(BLANK); header(s, title, kicker)
    if full:
        picture(s, FIGS[key], 0.4, 1.55, 12.5, 4.6); textbox(s, 0.6, 6.25, 12, 1.1, bullets, size=14)
    else:
        picture(s, FIGS[key], 0.4, 1.55, 8.6, 5.6); textbox(s, 9.2, 1.7, 3.8, 5.4, bullets, size=15)

s = prs.slides.add_slide(BLANK)
bg = s.shapes.add_shape(1, 0, 0, prs.slide_width, prs.slide_height); bg.fill.solid(); bg.fill.fore_color.rgb = RGBColor(0x3B, 0x10, 0x25); bg.line.fill.background()
textbox(s, .8, 2.2, 11.5, 1.2, ["A second opinion on BIOMASS forest height"], size=40, color=RGBColor(255, 255, 255), bold_first=True)
textbox(s, .8, 3.5, 11.5, 1.5, ["Official L2A and workshop FH against GEDI and NASA LVIS LiDAR at Rabi, Mabounie and Lopé (Gabon)",
                                "ESA MAAP Hackathon 2026 — Challenge #5 — generated automatically by the notebook"], size=18, color=RGBColor(0xF3, 0xDD, 0xE7))
fig_slide("The workshop product", "One geocoded FH map per acquisition (Nov 2025 – Jun 2026) + a mosaic", "01_product",
          ["No metadata: orbit from acquisition time, geometry from the valid-data footprint", "Put on the same grid as the official product and the LiDAR"])
fig_slide("Same forest, three maps", "Same colour scale, same window", "02_first_look", ["Official: smooth, narrow range", "Workshop: grainy, wide range"], full=False)
fig_slide("Two kinds of error", "Forest height distributions under the LiDAR", "03_histograms",
          ["Official L2A: too narrow (compressed, biased low on tall forest)", "Workshop: centred but too wide (noisy)"])
fig_slide("Pixel by pixel, the official product wins", "RMSE against LVIS at 100 m, by landscape stratum", "04_rmse_by_stratum", VERDICT[:1])
fig_slide("Noise or bias? The averaging test", "Product and LiDAR averaged over larger windows", "05_error_vs_scale", VERDICT[1:3])
fig_slide("Slopes facing the radar", "Forest error vs local incidence angle; solid = official, dashed = workshop", "06_error_vs_lia",
          [l for l in VERDICT if l.startswith("Slopes")][:3])
fig_slide("Is GEDI a good referee?", "GEDI − LVIS in forest pixels", "08_gedi_vs_lvis", [l for l in VERDICT if l.startswith("GEDI as referee")][:1])
fig_slide("BIOMASS against GEDI and against LVIS", "Solid: vs LVIS, dashed: vs GEDI, dotted: GEDI vs LVIS", "09_gedi_vs_lvis_referee",
          [l for l in VERDICT if l.startswith("Official L2A bias") or l.startswith("Against GEDI")][:2])
s = prs.slides.add_slide(BLANK); header(s, "Verdict", "Generated from the notebook results")
textbox(s, .6, 1.5, 12.1, 5.9, [f"{i}. {l}" for i, l in enumerate(VERDICT, 1)], size=11)
report = ROOT / "results" / "Workshop_FH_vs_L2A.pptx"; prs.save(report)
print("Report:", report, f"({len(prs.slides)} slides)")